In [ ]:
import sys
import os
import sys

sys.path.append(".../ProPepX_complete_code/")

from Import_libraries import *
from Dataset_Preprocessing import H5PairDataset
from pepPI_binding_site_collate import collate_fn
from CoBindingCNN import ProPepX
from compute_loss import compute_loss
from model_load_weight_utils import load_weight
from propepx_metrics import (
    compute_prot_metrics,
    compute_pep_pair_metrics,
    summarize_pep_pair_metrics,compute_joint_mode_metrics,
)
from propepx_inference_utils import (
    get_device,
    build_model,
    make_loader,
    evaluate_protein,
)

from model_load_weight_utils import load_weight

In [ ]:
MODE = "prot"
THRESHOLD = 0.50
BATCH_SIZE = 32

device = get_device("0")
print("Using device:", device)


EXPERIMENTS = [
    {
        "dataset": "TS092",
        "embedding": "ProtT5",
        "emb_dim": 1024,
        "weight": ".../Fine_tine_TS092_ProPepX_weight.pt",
        "test_h5": ".../Test_Dataset_TS092_embedding.h5",
    },
    {
        "dataset": "TS092",
        "embedding": "ESM-3",
        "emb_dim": 1152,
        "weight": ".../Fine_tine_TS092_ProPepX_weight.pt",
        "test_h5": ".../ESM-3_Test_Dataset_TS092_embedding.h5",
    },
    {
        "dataset": "TS125",
        "embedding": "ProtT5",
        "emb_dim": 1024,
        "weight": ".../Fine_tine_TS125_ProPepX_weight.pt",
        "test_h5": ".../Test_Dataset_TS125_embedding.h5",
    },
    {
        "dataset": "TS125",
        "embedding": "ESM-3",
        "emb_dim": 1152,
        "weight": ".../ESM-3_Test_Dataset_TS125_embedding.pt",
        "test_h5": ".../ESM-3_Test_Dataset_TS125_embedding.h5",
    },
    {
        "dataset": "TS251",
        "embedding": "ProtT5",
        "emb_dim": 1024,
        "weight": ".../Fine_tine_TS251_ProPepX_weight.pt",
        "test_h5": ".../Test_Dataset_TS251_embedding.h5",
    },
    {
        "dataset": "TS251",
        "embedding": "ESM-3",
        "emb_dim": 1152,
        "weight": ".../ESM-3_Test_Dataset_TS251_embedding.pt",
        "test_h5": ".../ESM-3_Test_Dataset_TS251_embedding.h5",
    },
    {
        "dataset": "TS639",
        "embedding": "ProtT5",
        "emb_dim": 1024,
        "weight": ".../Fine_tine_TS639_ProPepX_weight.pt",
        "test_h5": ".../Test_Dataset_TS639_embedding.h5",
    },
    {
        "dataset": "TS639",
        "embedding": "ESM-3",
        "emb_dim": 1152,
        "weight": ".../ESM-3_Test_Dataset_TS639_embedding.pt",
        "test_h5": ".../ESM-3_Test_Dataset_TS639_embedding.h5",
    },
]


all_results = []

for exp in EXPERIMENTS:
    print("\n" + "=" * 90)
    print(f"Dataset   : {exp['dataset']}")
    print(f"Embedding : {exp['embedding']}")
    print("=" * 90)

    try:
        dataset, loader = make_loader(
            test_h5=exp["test_h5"],
            mode=MODE,
            batch_size=BATCH_SIZE,
            num_workers=4,
            pin_memory=True,
        )

        model = build_model(
            emb_dim=exp["emb_dim"],
            mode=MODE,
            device=device,
            max_len_prot=1418,
            max_len_pep=50,
        )

        model = load_weight(model, exp["weight"], device)

        metrics = evaluate_protein(
            model=model,
            loader=loader,
            device=device,
            threshold=THRESHOLD,
        )

        row = {
            "Dataset": exp["dataset"],
            "Embedding": exp["embedding"],
            "Mode": MODE,
            "Threshold": THRESHOLD,
            "Test_Samples": len(dataset),
            **metrics,
            "Weight_Path": exp["weight"],
            "Test_H5": exp["test_h5"],
        }

        all_results.append(row)

        print("\nRESULT")
        for k in ["AUC", "AUPR", "MCC", "F1", "Recall", "Specificity", "Precision", "Accuracy"]:
            print(f"{k:15s}: {row[k]:.4f}")

    except Exception as e:
        print("FAILED:", exp["dataset"], exp["embedding"])
        print("Error:", repr(e))

        all_results.append({
            "Dataset": exp["dataset"],
            "Embedding": exp["embedding"],
            "Mode": MODE,
            "Threshold": THRESHOLD,
            "Status": "FAILED",
            "Error": repr(e),
            "Weight_Path": exp["weight"],
            "Test_H5": exp["test_h5"],
        })

    finally:
        if "model" in locals():
            del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


results_df = pd.DataFrame(all_results)
results_df.to_csv("protein_mode_results.csv", index=False)

print("\nFINAL SUMMARY")
display_cols = [
    "Dataset", "Embedding", "AUC", "AUPR", "MCC", "F1",
    "Recall", "Specificity", "Precision", "Accuracy"
]
print(results_df[display_cols].to_string(index=False))

Using device: cuda

Dataset   : TS092
Embedding : ProtT5
Loading ProPepX architecture in mode=prot
Loaded checkpoint: /home/kumail/Transformer Model/updatedModel-pepPI/Transfer Learning/Model_10M_parameters/Fine_Tune_TS092/Fine_Tune_TS092/best_finetune_prot.pt
Missing keys   : 0
Unexpected keys: 0



RESULT
AUC            : 0.9193
AUPR           : 0.6883
MCC            : 0.6094
F1             : 0.6503
Recall         : 0.6310
Specificity    : 0.9622
Precision      : 0.6709
Accuracy       : 0.9261

Dataset   : TS092
Embedding : ESM-3
Loading ProPepX architecture in mode=prot
Loaded checkpoint: /home/kumail/Transformer Model/updatedModel-pepPI/Transfer Learning/ESM embedding Model TL/Model_18.6M_parameters/Fine_Tune_TS092/best_finetune_prot.pt
Missing keys   : 0
Unexpected keys: 0



RESULT
AUC            : 0.8957
AUPR           : 0.6670
MCC            : 0.5952
F1             : 0.6162
Recall         : 0.5109
Specificity    : 0.9820
Precision      : 0.7760
Accuracy       : 0.9307

Dataset   : TS125
Embedding : ProtT5
Loading ProPepX architecture in mode=prot
Loaded checkpoint: /home/kumail/Transformer Model/updatedModel-pepPI/Transfer Learning/Model_10M_parameters/Fine_Tune_TS125/Fine_Tune_TS125/best_finetune_prot.pt
Missing keys   : 0
Unexpected keys: 0



RESULT
AUC            : 0.9251
AUPR           : 0.5788
MCC            : 0.5836
F1             : 0.6070
Recall         : 0.6428
Specificity    : 0.9720
Precision      : 0.5751
Accuracy       : 0.9537

Dataset   : TS125
Embedding : ESM-3
Loading ProPepX architecture in mode=prot
Loaded checkpoint: /home/kumail/Transformer Model/updatedModel-pepPI/Transfer Learning/ESM embedding Model TL/Model_18.6M_parameters/Fine_Tune_TS125/best_finetune_prot.pt
Missing keys   : 0
Unexpected keys: 0



RESULT
AUC            : 0.9190
AUPR           : 0.5634
MCC            : 0.5468
F1             : 0.5631
Recall         : 0.4953
Specificity    : 0.9845
Precision      : 0.6523
Accuracy       : 0.9573

Dataset   : TS251
Embedding : ProtT5
Loading ProPepX architecture in mode=prot
Loaded checkpoint: /home/kumail/Transformer Model/updatedModel-pepPI/Transfer Learning/Model_10M_parameters/Fine_Tune_TS251/Fine_Tune_TS251/best_finetune_prot.pt
Missing keys   : 0
Unexpected keys: 0



RESULT
AUC            : 0.9098
AUPR           : 0.7386
MCC            : 0.6778
F1             : 0.7124
Recall         : 0.6879
Specificity    : 0.9690
Precision      : 0.7386
Accuracy       : 0.9373

Dataset   : TS251
Embedding : ESM-3
Loading ProPepX architecture in mode=prot
Loaded checkpoint: /home/kumail/Transformer Model/updatedModel-pepPI/Transfer Learning/ESM embedding Model TL/Model_18.6M_parameters/Fine_Tune_TS251/best_finetune_prot.pt
Missing keys   : 0
Unexpected keys: 0



RESULT
AUC            : 0.8948
AUPR           : 0.6968
MCC            : 0.6273
F1             : 0.6583
Recall         : 0.5838
Specificity    : 0.9758
Precision      : 0.7546
Accuracy       : 0.9316

Dataset   : TS639
Embedding : ProtT5
Loading ProPepX architecture in mode=prot
Loaded checkpoint: /home/kumail/Transformer Model/updatedModel-pepPI/Transfer Learning/Model_10M_parameters/Fine_Tune_TS639/Fine_Tune_TS639/best_finetune_prot.pt
Missing keys   : 0
Unexpected keys: 0



RESULT
AUC            : 0.9029
AUPR           : 0.5422
MCC            : 0.5575
F1             : 0.5828
Recall         : 0.5915
Specificity    : 0.9738
Precision      : 0.5744
Accuracy       : 0.9522

Dataset   : TS639
Embedding : ESM-3
Loading ProPepX architecture in mode=prot
Loaded checkpoint: /home/kumail/Transformer Model/updatedModel-pepPI/Transfer Learning/ESM embedding Model TL/Model_18.6M_parameters/Fine_Tune_TS639/best_finetune_prot.pt
Missing keys   : 0
Unexpected keys: 0



RESULT
AUC            : 0.9070
AUPR           : 0.5206
MCC            : 0.5171
F1             : 0.5380
Recall         : 0.4840
Specificity    : 0.9811
Precision      : 0.6055
Accuracy       : 0.9530

FINAL SUMMARY
Dataset Embedding      AUC     AUPR      MCC       F1   Recall  Specificity  Precision  Accuracy
  TS092    ProtT5 0.919339 0.688340 0.609425 0.650327 0.630995     0.962179   0.670881  0.926121
  TS092     ESM-3 0.895747 0.666960 0.595168 0.616157 0.510900     0.981985   0.776039  0.930695
  TS125    ProtT5 0.925117 0.578808 0.583568 0.607045 0.642774     0.972045   0.575078  0.953741
  TS125     ESM-3 0.918957 0.563362 0.546754 0.563100 0.495338     0.984462   0.652341  0.957272
  TS251    ProtT5 0.909846 0.738633 0.677750 0.712363 0.687925     0.969011   0.738601  0.937274
  TS251     ESM-3 0.894756 0.696796 0.627303 0.658316 0.583799     0.975840   0.754640  0.931575
  TS639    ProtT5 0.902863 0.542230 0.557544 0.582835 0.591519     0.973766   0.574402  0.952178
  TS639  

In [ ]:
from propepx_inference_utils import (
    get_device,
    build_model,
    make_loader,
    evaluate_peptide,
)

from model_load_weight_utils import load_weight
MODE = "pep"
THRESHOLD = 0.50
BATCH_SIZE = 32

device = get_device("0")
print("Using device:", device)


EXPERIMENTS = [
    {
        "dataset": "CAMP_TS231",
        "embedding": "ProtT5",
        "emb_dim": 1024,
        "weight": ".../best_finetune_pep_mode_ProPepX_weight.pt",
        "test_h5": ".../CAMP_Test_ProttranT5_embeddings.h5",
    },
    {
        "dataset": "CAMP_TS231",
        "embedding": "ESM-3",
        "emb_dim": 1152,
        "weight": ".../best_finetune_pep_mode_ProPepX_weight.pt",
        "test_h5": ".../CAMP_Test_ESM-600_embeddings.h5",
    },
]


all_results = []

for exp in EXPERIMENTS:
    print("\n" + "=" * 80)
    print(f"Dataset   : {exp['dataset']}")
    print(f"Embedding : {exp['embedding']}")
    print("=" * 80)

    try:
        dataset, loader = make_loader(
            test_h5=exp["test_h5"],
            mode=MODE,
            batch_size=BATCH_SIZE,
            num_workers=0,
            pin_memory=False,
        )

        model = build_model(
            emb_dim=exp["emb_dim"],
            mode=MODE,
            device=device,
            max_len_prot=1418,
            max_len_pep=50,
        )

        model = load_weight(model, exp["weight"], device)

        summary, per_sample_df = evaluate_peptide(
            model=model,
            loader=loader,
            device=device,
            threshold=THRESHOLD,
        )

        per_sample_file = f"per_sample_peptide_{exp['dataset']}_{exp['embedding']}.tsv"
        per_sample_df.to_csv(per_sample_file, sep="\t", index=False)

        row = {
            "Dataset": exp["dataset"],
            "Embedding": exp["embedding"],
            "Mode": MODE,
            "Threshold": THRESHOLD,
            "Test_Samples": len(dataset),
            **summary,
            "Per_Sample_File": per_sample_file,
            "Weight_Path": exp["weight"],
            "Test_H5": exp["test_h5"],
        }

        all_results.append(row)

        print("\nMEAN PER-PAIR RESULT")
        for k in [
            "Mean_AUC", "Mean_MCC", "Mean_Specificity",
            "Mean_Precision", "Mean_Recall", "Mean_F1", "Mean_Accuracy"
        ]:
            print(f"{k:22s}: {row[k]:.4f}")

    except Exception as e:
        print("FAILED:", exp["dataset"], exp["embedding"])
        print("Error:", repr(e))

        all_results.append({
            "Dataset": exp["dataset"],
            "Embedding": exp["embedding"],
            "Mode": MODE,
            "Threshold": THRESHOLD,
            "Status": "FAILED",
            "Error": repr(e),
            "Weight_Path": exp["weight"],
            "Test_H5": exp["test_h5"],
        })

    finally:
        if "model" in locals():
            del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


results_df = pd.DataFrame(all_results)
results_df.to_csv("peptide_mode_results.csv", index=False)

print("\nFINAL SUMMARY")
display_cols = [
    "Dataset",
    "Embedding",
    "Mean_AUC",
    "Mean_MCC",
    "Mean_Specificity",
    "Mean_Precision",
    "Mean_Recall",
    "Mean_F1",
    "Mean_Accuracy",
]
print(results_df[display_cols].to_string(index=False))

Using device: cuda

Dataset   : CAMP_TS231
Embedding : ProtT5
Loading ProPepX architecture in mode=pep
Loaded checkpoint: /home/kumail/Transformer Model/updatedModel-pepPI/Transfer Learning/peptide_binding_sites_TL_model/ProttransT5/Model_18.5M_parameters/Fine_Tune_pepTS231/best_finetune_pep.pt
Missing keys   : 0
Unexpected keys: 0



MEAN PER-PAIR RESULT
Mean_AUC              : 0.8079
Mean_MCC              : 0.6339
Mean_Specificity      : 0.7055
Mean_Precision        : 0.8291
Mean_Recall           : 0.9214
Mean_F1               : 0.8603
Mean_Accuracy         : 0.8297

Dataset   : CAMP_TS231
Embedding : ESM-3
Loading ProPepX architecture in mode=pep
Loaded checkpoint: /home/kumail/Transformer Model/updatedModel-pepPI/Transfer Learning/peptide_binding_sites_TL_model/ESM-600/Model_18.75M_parameters/Fine_Tune_pepTS231/best_finetune_pep.pt
Missing keys   : 0
Unexpected keys: 0



MEAN PER-PAIR RESULT
Mean_AUC              : 0.7975
Mean_MCC              : 0.6203
Mean_Specificity      : 0.6858
Mean_Precision        : 0.8236
Mean_Recall           : 0.9306
Mean_F1               : 0.8591
Mean_Accuracy         : 0.8243

FINAL SUMMARY
   Dataset Embedding  Mean_AUC  Mean_MCC  Mean_Specificity  Mean_Precision  Mean_Recall  Mean_F1  Mean_Accuracy
CAMP_TS231    ProtT5  0.807880  0.633941          0.705461        0.829083     0.921448 0.860270       0.829737
CAMP_TS231     ESM-3  0.797485  0.620305          0.685770        0.823633     0.930558 0.859127       0.824322


In [ ]:
from propepx_inference_utils import (
    get_device,
    build_model,
    make_loader,
    evaluate_joint,
    mean_sd,
)

from model_load_weight_utils import load_weight


MODE = "mode-GLOBAL"
MODEL_MODE = "mode-GLOBAL"
THRESHOLD = 0.50
BATCH_SIZE = 16

device = get_device("0")
print("Using device:", device)


EXPERIMENTS = [
    {
        "dataset": "Test167",
        "embedding": "ProtTransT5",
        "emb_dim": 1024,
        "test_h5": ".../Test167_embeddings.h5",
        "fold_weights": [
            ".../best_fold_1_joint_mode_ProPepX.pt",
            ".../best_fold_2_joint_mode_ProPepX.pt",
            ".../best_fold_3_joint_mode_ProPepX.pt",
            ".../best_fold_4_joint_mode_ProPepX.pt",
            ".../best_fold_5_joint_mode_ProPepX.pt",
        ],
    },
    {
        "dataset": "Test167",
        "embedding": "ESM-3",
        "emb_dim": 1152,
        "test_h5": ".../Test167_ESM-600M_embeddings.h5",
        "fold_weights": [
            ".../best_fold_1_joint_mode_ProPepX.pt",
            ".../best_fold_2_joint_mode_ProPepX.pt",
            ".../best_fold_3_joint_mode_ProPepX.pt",
            ".../best_fold_4_joint_mode_ProPepX.pt",
            ".../best_fold_5_joint_mode_ProPepX.pt",
        ],
    },
    {
        "dataset": "Test251_LEADSPEP",
        "embedding": "ProtTransT5",
        "emb_dim": 1024,
        "test_h5": ".../Test251_LEADSPEP_embedding.h5",
        "fold_weights": [
            ".../best_fold_1_joint_mode_ProPepX.pt",
            ".../best_fold_2_joint_mode_ProPepX.pt",
            ".../best_fold_3_joint_mode_ProPepX.pt",
            ".../best_fold_4_joint_mode_ProPepX.pt",
            ".../best_fold_5_joint_mode_ProPepX.pt",
        ],
    },
    {
        "dataset": "Test251_LEADSPEP",
        "embedding": "ESM-3",
        "emb_dim": 1152,
        "test_h5": ".../Test251_LEADSPEP_ESM-600M_embeddings.h5",
        "fold_weights": [
            ".../best_fold_1_joint_mode_ProPepX.pt",
            ".../best_fold_2_joint_mode_ProPepX.pt",
            ".../best_fold_3_joint_mode_ProPepX.pt",
            ".../best_fold_4_joint_mode_ProPepX.pt",
            ".../best_fold_5_joint_mode_ProPepX.pt",
        ],
    },
]


all_experiment_rows = []
all_summary_rows = []


for exp in EXPERIMENTS:
    print("\n" + "#" * 120)
    print(f"RUNNING EXPERIMENT: {exp['dataset']} | {exp['embedding']}")
    print("#" * 120)

    try:
        test_dataset, test_loader = make_loader(
            test_h5=exp["test_h5"],
            mode=MODE,
            batch_size=BATCH_SIZE,
            num_workers=4,
            pin_memory=True,
        )

        print("Test samples :", len(test_dataset))
        print("Embedding dim:", exp["emb_dim"])
        print("Test H5      :", exp["test_h5"])

        experiment_rows = []

        for fold_id, weight_path in enumerate(exp["fold_weights"], start=1):
            print("\n" + "=" * 90)
            print(f"{exp['embedding']} | REPRODUCING FOLD {fold_id}")
            print("=" * 90)

            model = build_model(
                emb_dim=exp["emb_dim"],
                mode=MODEL_MODE,
                device=device,
                max_len_prot=1418,
                max_len_pep=50,
            )

            model = load_weight(model, weight_path, device)

            (
                test_loss,
                prot_metrics,
                pep_metrics,
                mean_mcc,
                mean_aupr,
                mean_auroc,
            ) = evaluate_joint(
                model=model,
                loader=test_loader,
                device=device,
                mode=MODE,
                threshold=THRESHOLD,
            )

            row = {
                "dataset": exp["dataset"],
                "embedding": exp["embedding"],
                "fold": fold_id,

                "test_prot_MCC": prot_metrics["MCC"],
                "test_prot_ACC": prot_metrics["ACC"],
                "test_prot_F1": prot_metrics["F1"],
                "test_prot_AUROC": prot_metrics["AUROC"],
                "test_prot_AUPR": prot_metrics["AUPR"],
                "test_prot_Positive": prot_metrics["Positive"],
                "test_prot_Negative": prot_metrics["Negative"],
                "test_prot_Total": prot_metrics["Total"],

                "test_pep_MCC": pep_metrics["MCC"],
                "test_pep_ACC": pep_metrics["ACC"],
                "test_pep_F1": pep_metrics["F1"],
                "test_pep_AUROC": pep_metrics["AUROC"],
                "test_pep_AUPR": pep_metrics["AUPR"],
                "test_pep_Positive": pep_metrics["Positive"],
                "test_pep_Negative": pep_metrics["Negative"],
                "test_pep_Total": pep_metrics["Total"],
            }

            experiment_rows.append(row)
            all_experiment_rows.append(row)

            print(f"{exp['embedding']} Fold {fold_id} Test Prot MCC : {prot_metrics['MCC']:.4f}")
            print(f"{exp['embedding']} Fold {fold_id} Test Pep  MCC : {pep_metrics['MCC']:.4f}")
            print(f"{exp['embedding']} Fold {fold_id} Test Mean MCC : {mean_mcc:.4f}")

            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        exp_df = pd.DataFrame(experiment_rows)

        summary_row = {
            "Dataset": exp["dataset"],
            "Embedding": exp["embedding"],

            "Protein AUROC": mean_sd(exp_df, "test_prot_AUROC"),
            "Protein AUPR": mean_sd(exp_df, "test_prot_AUPR"),
            "Protein MCC": mean_sd(exp_df, "test_prot_MCC"),
            "Protein F1": mean_sd(exp_df, "test_prot_F1"),
            "Protein ACC": mean_sd(exp_df, "test_prot_ACC"),

            "Peptide AUROC": mean_sd(exp_df, "test_pep_AUROC"),
            "Peptide AUPR": mean_sd(exp_df, "test_pep_AUPR"),
            "Peptide MCC": mean_sd(exp_df, "test_pep_MCC"),
            "Peptide F1": mean_sd(exp_df, "test_pep_F1"),
            "Peptide ACC": mean_sd(exp_df, "test_pep_ACC"),
        }

        all_summary_rows.append(summary_row)

    except Exception as e:
        print("FAILED EXPERIMENT:", exp["dataset"], exp["embedding"])
        print("Error:", repr(e))

        all_summary_rows.append({
            "Dataset": exp["dataset"],
            "Embedding": exp["embedding"],
            "Status": "FAILED",
            "Error": repr(e),
        })

    finally:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


all_results_df = pd.DataFrame(all_experiment_rows)
final_summary = pd.DataFrame(all_summary_rows)

print("\n" + "=" * 90)
print("FINAL SUMMARY: ProtTransT5 vs ESM-3")
print("=" * 90)
print(final_summary.to_string(index=False))

Using device: cuda

########################################################################################################################
RUNNING EXPERIMENT: Test167 | ProtTransT5
########################################################################################################################
Test samples : 246
Embedding dim: 1024
Test H5      : /home/kumail/Transformer Model/KGIPA Dataset/KGIPA_Prot_L1418/ProttransT5_embedding/Test167_embeddings.h5

ProtTransT5 | REPRODUCING FOLD 1
Loading ProPepX architecture in mode=both
Loaded checkpoint: /home/kumail/Transformer Model/updatedModel-pepPI/Transfer Learning/KCIPA_model_comparsion/Global_Predictions/Transfer Learning Global/Fine_Tune_KGIPA_Global/FiveFold_Global_FineTune/fold_1/best_fold_1_global.pt
Missing keys   : 0
Unexpected keys: 0


ProtTransT5 Fold 1 Test Prot MCC : 0.5447
ProtTransT5 Fold 1 Test Pep  MCC : 0.3877
ProtTransT5 Fold 1 Test Mean MCC : 0.4662

ProtTransT5 | REPRODUCING FOLD 2
Loading ProPepX architecture in mode=both
Loaded checkpoint: /home/kumail/Transformer Model/updatedModel-pepPI/Transfer Learning/KCIPA_model_comparsion/Global_Predictions/Transfer Learning Global/Fine_Tune_KGIPA_Global/FiveFold_Global_FineTune/fold_2/best_fold_2_global.pt
Missing keys   : 0
Unexpected keys: 0


ProtTransT5 Fold 2 Test Prot MCC : 0.5439
ProtTransT5 Fold 2 Test Pep  MCC : 0.3937
ProtTransT5 Fold 2 Test Mean MCC : 0.4688

ProtTransT5 | REPRODUCING FOLD 3
Loading ProPepX architecture in mode=both
Loaded checkpoint: /home/kumail/Transformer Model/updatedModel-pepPI/Transfer Learning/KCIPA_model_comparsion/Global_Predictions/Transfer Learning Global/Fine_Tune_KGIPA_Global/FiveFold_Global_FineTune/fold_3/best_fold_3_global.pt
Missing keys   : 0
Unexpected keys: 0


ProtTransT5 Fold 3 Test Prot MCC : 0.5473
ProtTransT5 Fold 3 Test Pep  MCC : 0.3946
ProtTransT5 Fold 3 Test Mean MCC : 0.4710

ProtTransT5 | REPRODUCING FOLD 4
Loading ProPepX architecture in mode=both
Loaded checkpoint: /home/kumail/Transformer Model/updatedModel-pepPI/Transfer Learning/KCIPA_model_comparsion/Global_Predictions/Transfer Learning Global/Fine_Tune_KGIPA_Global/FiveFold_Global_FineTune/fold_4/best_fold_4_global.pt
Missing keys   : 0
Unexpected keys: 0


ProtTransT5 Fold 4 Test Prot MCC : 0.5396
ProtTransT5 Fold 4 Test Pep  MCC : 0.4025
ProtTransT5 Fold 4 Test Mean MCC : 0.4711

ProtTransT5 | REPRODUCING FOLD 5
Loading ProPepX architecture in mode=both
Loaded checkpoint: /home/kumail/Transformer Model/updatedModel-pepPI/Transfer Learning/KCIPA_model_comparsion/Global_Predictions/Transfer Learning Global/Fine_Tune_KGIPA_Global/FiveFold_Global_FineTune/fold_5/best_fold_5_global.pt
Missing keys   : 0
Unexpected keys: 0


...

ESM-3 Fold 2 Test Prot MCC : 0.5811
ESM-3 Fold 2 Test Pep  MCC : 0.4939
ESM-3 Fold 2 Test Mean MCC : 0.5375

ESM-3 | REPRODUCING FOLD 3
Loading ProPepX architecture in mode=both
Loaded checkpoint: /home/kumail/Transformer Model/updatedModel-pepPI/Transfer Learning/KCIPA_model_comparsion/Global_Predictions/Transfer Learning Global/ESM-600M/Fine_Tune_KGIPA_Global/Test125_&_LEADS_fold_3/best_fold_3_global.pt
Missing keys   : 0
Unexpected keys: 0


ESM-3 Fold 3 Test Prot MCC : 0.5823
ESM-3 Fold 3 Test Pep  MCC : 0.5011
ESM-3 Fold 3 Test Mean MCC : 0.5417

ESM-3 | REPRODUCING FOLD 4
Loading ProPepX architecture in mode=both
Loaded checkpoint: /home/kumail/Transformer Model/updatedModel-pepPI/Transfer Learning/KCIPA_model_comparsion/Global_Predictions/Transfer Learning Global/ESM-600M/Fine_Tune_KGIPA_Global/Test125_&_LEADS_fold_4/best_fold_4_global.pt
Missing keys   : 0
Unexpected keys: 0


ESM-3 Fold 4 Test Prot MCC : 0.5764
ESM-3 Fold 4 Test Pep  MCC : 0.4953
ESM-3 Fold 4 Test Mean MCC : 0.5359

ESM-3 | REPRODUCING FOLD 5
Loading ProPepX architecture in mode=both
Loaded checkpoint: /home/kumail/Transformer Model/updatedModel-pepPI/Transfer Learning/KCIPA_model_comparsion/Global_Predictions/Transfer Learning Global/ESM-600M/Fine_Tune_KGIPA_Global/Test125_&_LEADS_fold_5/best_fold_5_global.pt
Missing keys   : 0
Unexpected keys: 0


ESM-3 Fold 5 Test Prot MCC : 0.5844
ESM-3 Fold 5 Test Pep  MCC : 0.5027
ESM-3 Fold 5 Test Mean MCC : 0.5435

FINAL SUMMARY: ProtTransT5 vs ESM-3
         Dataset   Embedding   Protein AUROC    Protein AUPR     Protein MCC      Protein F1     Protein ACC   Peptide AUROC    Peptide AUPR     Peptide MCC      Peptide F1     Peptide ACC
         Test167 ProtTransT5 0.9409 ± 0.0010 0.5394 ± 0.0024 0.5437 ± 0.0028 0.5546 ± 0.0024 0.9701 ± 0.0002 0.8020 ± 0.0022 0.6527 ± 0.0053 0.3945 ± 0.0053 0.5952 ± 0.0031 0.6869 ± 0.0044
         Test167       ESM-3 0.9400 ± 0.0011 0.5394 ± 0.0045 0.5395 ± 0.0047 0.5512 ± 0.0043 0.9702 ± 0.0003 0.8038 ± 0.0029 0.6467 ± 0.0066 0.4064 ± 0.0067 0.6022 ± 0.0039 0.6916 ± 0.0026
Test251_LEADSPEP ProtTransT5 0.9645 ± 0.0005 0.6101 ± 0.0052 0.6102 ± 0.0029 0.6221 ± 0.0028 0.9574 ± 0.0005 0.8193 ± 0.0016 0.7780 ± 0.0042 0.5020 ± 0.0065 0.7780 ± 0.0024 0.7252 ± 0.0036
Test251_LEADSPEP       ESM-3 0.9577 ± 0.0003 0.5873 ± 0.0025 0.5808 ± 0.0029 0.5984 ± 0.0026 0.9575

: 

In [ ]:
from propepx_inference_utils import (
    get_device,
    build_model,
    make_loader,
    evaluate_joint,
)

from model_load_weight_utils import load_weight


MODE = "mode-GLOBAL"
THRESHOLD = 0.50
BATCH_SIZE = 8

device = get_device("0")
print("Using device:", device)


EXPERIMENTS = [
    {
        "dataset": "TS167",
        "embedding": "ProtT5",
        "emb_dim": 1024,
        "weight": ".../bestbest_pretrain_zero_shot_mode_ProPepX.pt",
        "test_h5": ".../Test_Dataset_TS167_ProttransT5_embedding.h5",
    },
    {
        "dataset": "TS167",
        "embedding": "ESM-3",
        "emb_dim": 1152,
        "weight": ".../Test_Dataset_TS167_ESM-3_embedding.pt",
        "test_h5": ".../Test_Dataset_TS167_ESM-3_embedding.h5",
    },
]


all_rows = []

for exp in EXPERIMENTS:
    print("\n" + "=" * 100)
    print(f"RUNNING: {exp['dataset']} | {exp['embedding']}")
    print("=" * 100)

    try:
        dataset, loader = make_loader(
            test_h5=exp["test_h5"],
            mode=MODE,
            batch_size=BATCH_SIZE,
            num_workers=4,
            pin_memory=True,
        )

        model = build_model(
            emb_dim=exp["emb_dim"],
            mode=MODE,
            device=device,
            max_len_prot=8000,
            max_len_pep=50,
        )

        model = load_weight(model, exp["weight"], device)

        test_loss, prot_metrics, pep_metrics, mean_mcc, mean_aupr, mean_auroc = evaluate_joint(
            model=model,
            loader=loader,
            device=device,
            mode=MODE,
            threshold=THRESHOLD,
        )

        row = {
            "Dataset": exp["dataset"],
            "Embedding": exp["embedding"],

            "Protein AUROC": prot_metrics["AUROC"],
            "Protein AUPR": prot_metrics["AUPR"],
            "Protein MCC": prot_metrics["MCC"],
            "Protein F1": prot_metrics["F1"],
            "Protein ACC": prot_metrics["ACC"],

            "Peptide AUROC": pep_metrics["AUROC"],
            "Peptide AUPR": pep_metrics["AUPR"],
            "Peptide MCC": pep_metrics["MCC"],
            "Peptide F1": pep_metrics["F1"],
            "Peptide ACC": pep_metrics["ACC"],

        }

        all_rows.append(row)

        print("\nRESULT")
        print(f"Protein AUROC : {prot_metrics['AUROC']:.4f}")
        print(f"Protein AUPR  : {prot_metrics['AUPR']:.4f}")
        print(f"Protein MCC   : {prot_metrics['MCC']:.4f}")
        print(f"Protein F1    : {prot_metrics['F1']:.4f}")
        print(f"Protein ACC   : {prot_metrics['ACC']:.4f}")

        print(f"Peptide AUROC : {pep_metrics['AUROC']:.4f}")
        print(f"Peptide AUPR  : {pep_metrics['AUPR']:.4f}")
        print(f"Peptide MCC   : {pep_metrics['MCC']:.4f}")
        print(f"Peptide F1    : {pep_metrics['F1']:.4f}")
        print(f"Peptide ACC   : {pep_metrics['ACC']:.4f}")

    except Exception as e:
        print("FAILED:", exp["dataset"], exp["embedding"])
        print("Error:", repr(e))

        all_rows.append({
            "Dataset": exp["dataset"],
            "Embedding": exp["embedding"],
            "Status": "FAILED",
            "Error": repr(e),
            "Weight_Path": exp["weight"],
            "Test_H5": exp["test_h5"],
        })

    finally:
        if "model" in locals():
            del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


final_df = pd.DataFrame(all_rows)
final_df.to_csv("zeroshot_mode_results.csv", index=False)

print("\nFINAL SUMMARY: SINGLE WEIGHT TEST RESULTS")
print(final_df.to_string(index=False))

Using device: cuda

RUNNING: TS167 | ProtT5
Loading ProPepX architecture in mode=both
Loaded checkpoint: /home/kumail/Transformer Model/updatedModel-pepPI/Transfer Learning/Both mode Global Predictions/ProttransT5_checkpoint/best_model_mode-GLOBAL.pt
Missing keys   : 0
Unexpected keys: 0



RESULT
Protein AUROC : 0.9472
Protein AUPR  : 0.6328
Protein MCC   : 0.6050
Protein F1    : 0.6197
Protein ACC   : 0.9709
Peptide AUROC : 0.8714
Peptide AUPR  : 0.8977
Peptide MCC   : 0.6131
Peptide F1    : 0.8754
Peptide ACC   : 0.8289

RUNNING: TS167 | ESM-3
Loading ProPepX architecture in mode=both
Loaded checkpoint: /home/kumail/Transformer Model/updatedModel-pepPI/Transfer Learning/Both mode Global Predictions/ESM-3_checkpoint/best_model_mode-GLOBAL.pt
Missing keys   : 0
Unexpected keys: 0



RESULT
Protein AUROC : 0.9287
Protein AUPR  : 0.6090
Protein MCC   : 0.5814
Protein F1    : 0.5911
Protein ACC   : 0.9730
Peptide AUROC : 0.8796
Peptide AUPR  : 0.9226
Peptide MCC   : 0.5839
Peptide F1    : 0.8643
Peptide ACC   : 0.8158

FINAL SUMMARY: SINGLE WEIGHT TEST RESULTS
Dataset Embedding  Protein AUROC  Protein AUPR  Protein MCC  Protein F1  Protein ACC  Peptide AUROC  Peptide AUPR  Peptide MCC  Peptide F1  Peptide ACC
  TS167    ProtT5       0.947188      0.632820     0.605031    0.619669     0.970933       0.871391      0.897707     0.613138    0.875438     0.828874
  TS167     ESM-3       0.928684      0.609015     0.581363    0.591112     0.972985       0.879577      0.922611     0.583889    0.864275     0.815784
